In [5]:
import zipfile
import geopandas as gpd
import pandas as pd

# Paths
gtfs_path = r"C:/Users/mpanta/Documents/AT3_KGG375/tas_gtfs.zip"
output_gpkg = r"C:/Users/mpanta/Documents/AT3_KGG375/burnie_transit_stops.gpkg"

# 1. Load GTFS data
with zipfile.ZipFile(gtfs_path, "r") as z:
  stops = pd.read_csv(z.open("stops.txt"), dtype={"stop_id": str})
  stop_times = pd.read_csv(
      z.open("stop_times.txt"), dtype={"stop_id": str, "trip_id": str}
  )

# 2. Filter Burnie stops and save spatial layer for QGIS
burnie_stops = stops[
    stops["stop_name"].str.contains("Burnie", case=False, na=False)
]
burnie_stop_ids = burnie_stops["stop_id"].unique()

gdf_stops = gpd.GeoDataFrame(
    burnie_stops,
    geometry=gpd.points_from_xy(
        burnie_stops["stop_lon"], burnie_stops["stop_lat"]
    ),
    crs="EPSG:4326",
)
gdf_stops.to_file(output_gpkg, layer="burnie_stops", driver="GPKG")
print(f"Spatial stops saved to: {output_gpkg}")

# 3. Merge stop times with full stop names
burnie_stop_times = stop_times[stop_times["stop_id"].isin(burnie_stop_ids)]
trip_schedule = burnie_stop_times.merge(
    stops[["stop_id", "stop_name"]], on="stop_id"
)


# 4. Function to get travel time between Point A and Point B
def get_travel_time(schedule_df, stop_a_name, stop_b_name):
  a_rows = schedule_df[
      schedule_df["stop_name"].str.contains(stop_a_name, case=False, na=False)
  ]
  b_rows = schedule_df[
      schedule_df["stop_name"].str.contains(stop_b_name, case=False, na=False)
  ]

  common_trips = set(a_rows["trip_id"]).intersection(set(b_rows["trip_id"]))
  results = []

  for trip_id in common_trips:
    t_a = a_rows[a_rows["trip_id"] == trip_id].iloc[0]
    t_b = b_rows[b_rows["trip_id"] == trip_id].iloc[0]

    # Check chronological direction (Stop A before Stop B)
    if t_a["stop_sequence"] < t_b["stop_sequence"]:
      dep_time = pd.to_timedelta(t_a["departure_time"])
      arr_time = pd.to_timedelta(t_b["arrival_time"])
      duration = (arr_time - dep_time).total_seconds() / 60

      results.append({
          "trip_id": trip_id,
          "from": t_a["stop_name"],
          "to": t_b["stop_name"],
          "departure": t_a["departure_time"],
          "arrival": t_b["arrival_time"],
          "duration_mins": round(duration, 2),
      })

  return pd.DataFrame(results)


# Example usage: Find travel times between two specific locations
df_travel = get_travel_time(trip_schedule, "Burnie", "Somerset")
print(df_travel.head(10))

Spatial stops saved to: C:/Users/mpanta/Documents/AT3_KGG375/burnie_transit_stops.gpkg
Empty DataFrame
Columns: []
Index: []
